# धडा 18 (पुढील): पावत्या ज्या सिद्ध करतात की *मानव* ने क्रिया अधिकृत केली

हा धडा काय दाखवतो की **एजंट** ने काय केले आणि **गेट** ने काय ठरवले. हा नोटबुक हरवलेला अर्धा भाग जोडतो: पुरावा की **नाविन व्यक्ती** ने **अचूक** क्रियेला मान्यता दिली आहे — संपूर्ण कॅनॉनिकल क्रियेवर मानवी स्वाक्षरी, जी ऑफलाइन पडताळली जाते.

इथे दोन्ही आर्टिफॅक्ट्स धड्याच्या पावत्यांसारख्या **त्याच एन्ब्लोप स्वरूपाचा** वापर करतात: एक फ्लॅट पेलोड ज्यात `type` फील्ड असतो, Ed25519 ने SHA-256 कॅनॉनिकल पेलोड बाइट्सवर स्वाक्षरी केली जाते, ज्याच्यावर संरचित `signature` ऑब्जेक्ट जोडलेले असते (आणि म्हणून स्वाक्षरी केलेल्या बाइट्सपासून वगळलेले असते). मंजुरीची पावती नवीन `type` (`human.approval.v1`) आहे जी क्रिया प्रकारासोबत आहे, त्यामुळे एक `verify_chain` मुख्य नोटबुकमध्ये आपण बांधलेल्या कोडच्या मार्गाने दोन्ही प्रकारच्या आर्टिफॅक्ट्ससाठी लागू होते. हेच इंटरनेट-ड्राफ्टमधील सह-स्वाक्षरी मार्गाचा स्वरूपही आहे ज्याचे हे धडे अनुसरतो (draft-farley-acta-signed-receipts).

मुख्य नोटबुकमधील डेमो व्हेरिफायरच्या तुलनेत एक संकल्पित सुधारणा: व्हेरिफायर `signature.key_id` ची तपासणी **पिन केलेल्या की नोंदणीविरुद्ध** करतो, पावतीत अंतर्भूत असलेल्या सार्वजनिक कीवर अवलंबून न राहता. हेच उत्पादन स्थिती आहे ज्याची धड्याच्या स्वतःच्या चेकलिस्टमध्ये शिफारस केली आहे ("व्हेरिफिकेशन सार्वजनिक की प्रकाशित करा"), आणि यामुळे बनावट आणणे नाकारले जाते, की स्वतःची की वापरण्याचा मार्ग बंद होतो.

हा नोटबुक शिकवणारा नियम: **स्वाक्षरी केलेली मंजुरी स्वतःमध्ये अधिकार नाही.** अधिकार फक्त तेव्हा अस्तित्वात असतो जेव्हा मंजुरीची पावती आणि क्रियेची पावती अंमलबजावणीच्या वेळी अजूनही त्याच कॅनॉनिकल कृतीशी बांधली जातात, ज्या धोरण आवृत्ती, की, आणि कालबाह्यता अद्याप चालू असते, आणि मंजुरी अजून वापरलेली नसते. प्रत्येक अपयश वेगवेगळ्या कारणांसह नाकारले जाते, त्यामुळे तुम्हाला *अधिकार कालबाह्य झाला* वेगळे पटते किंवा *अमलबजावणी झालेली क्रिया बदलली* हे सांगू शकता.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## अचूक क्रिया

मान्यतेचे एकक म्हणजे **कॅनॉनिकल क्रिया ऑब्जेक्ट** — "परतावा मंजूर करा" सारखा अस्पष्ट लेबल नाही, तर नेमकी, पूर्णपणे निर्दिष्ट केलेली क्रिया. संपूर्ण ऑब्जेक्टवर सह्य करणे (आणि त्यातून एक डाइजेस्ट तयार करणे) या गोष्टीमुळे आपण नंतर पुरावा देऊ शकतो की मानवीने *ही*च क्रिया मंजूर केली आणि काहीही नाही.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## एक कापूस, दोन अधिकृत संस्था

प्रत्येक पावती ही धड्याची कापूस आहे: `type` क्षेत्र असलेली एक सपाट पेलोड, तसेच `signature` ऑब्जेक्ट (`alg`, `sig`, `key_id`) जे सही केलेल्या बाइट्सचा भाग नाही. `verify_envelope` हा दोन्ही पावती प्रकारांसाठी सामायिक संरचनात्मक + सही तपासणी आहे; जे **पिन केलेले कळ नोंदणी** मध्ये ते `signature.key_id` सोडवते ते अधिकार वेगळे ठेवते:

- **मंजुरी पावती** (`human.approval.v1`) — नाविक मंजूर करणारा, संपूर्ण प्रामाणिक क्रिया **आणि त्याचा डाइजेस्ट**, `policy_version`, निर्गमन + कालबाह्यता टाईमस्टँप. एकदाच वापर साखळी पातळीवर नोंदवले जाते.
- **क्रिया पावती** (`agent.action.v1`) — एजंट ओळख, `run_id`, तोच प्रामाणिक क्रिया **डाइजेस्ट**, अंमलबजावणीचा परिणाम + टाईमस्टँप, आणि `parent_approval_ref`: मंजुरीची `receipt_hash`, धड्याच्या साखळीतल्या `previous_receipt_hash` प्रमाणेच.

सामायिक `action_digest` क्षेत्र हा जोडणीचा आधार आहे. `key_id` सही ऑब्जेक्टमध्ये एक लुकअप हिंट म्हणून राहतो: त्याला वेगळ्या पिन केलेल्या कळीवर पुनर्निर्देशित केल्यास सही तपासणी अयशस्वी होते, त्यामुळे त्याचा अर्थ काहीही जात नाही.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: जिथे बांधणी प्रत्यक्ष ठरवली जाते

`verify_chain` दोन सही तपासण्यांवर आधारित सोयीस्कर आवरण **नाही**. हे एकमेव ठिकाण आहे जिथे सामायिक अधिकृत `action_digest`, मंजुरीची धोरण/की/कालमर्यादा **ताजेपणा**, आणि मंजुरीचे **एकदाच वापर** एकत्र तपासले जातात, अंमलात आणल्या जाणाऱ्या क्रियाविषयी *आत्ताच*.

प्रत्येक अयशस्वीतीसह एक **वेगळा कारण** दिला जातो, ज्यामुळे नाकारण्याचा वाचक जाणू शकतो की अधिकार जुने झाले आहेत (धोरण बदलले, की फिरवले, मंजुरी कालबाह्य झाली, मंजुरी वापरली गेली) किंवा अंमलबजावणी करत असलेली कृती अद्याप वैध मंजुरीच्या आत बदलली आहे (डाइजेस्ट बदल). 


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## बाइंडिंग काय पकडते

खालील प्रत्येक प्रकरण **तडकलेले** असते एका **वेगळ्या कारणाने**. पहिला ब्लॉक हा क्लासिक संच आहे (तडजोड, गोंधळलेला डेप्युटी, रिप्ले, प्राधिकरणावर बनावट, खराब स्वरूपाचा इनपुट). दुसरा ब्लॉक हा जोडी आहे जो मालमत्तेला असलेल्या ऐवजी खरी बनवतो:

- **जुनी प्राधिकरण** — स्वाक्षरी अद्याप वैध आहे, पण धोरण आवृत्ती बदलली आहे, मंजूर करणाऱ्याची की पिन केलेल्या नोंदणीतून फिरवलेली आहे, किंवा मंजुरी अंमलबजावणीपूर्वी कालबाह्य झाली आहे;
- **डाइजेस्ट स्थानापन्न** — वैध स्वाक्षरी असलेल्या क्रियेचा पावती जो `parent_approval_ref` एक *खऱ्या* मंजुरीकडे निर्देशित करतो, पण त्या मंजुरीचा कानोनी क्रिया डाइजेस्ट प्रत्यक्षात चालवण्यात आलेल्या क्रियेशी जुळत नाही.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## हे काय सिद्ध करते — आणि काय करत नाही

**सिद्ध करते:** एक नाविन्यपूर्ण मान्यताप्राप्त मनुष्य *हे अचूक पंचायती क्रिया* (पूर्ण क्रिया + डाइजेस्ट, कीशी स्वाक्षरी केलेली जी पिन केलेल्या नोंदणीतून सादर केलेली आहे), आणि एजंटने *नक्कीच तीच मान्यताप्राप्त क्रिया* (समान डाइजेस्ट, `receipt_hash` ने मान्यतेशी गांठलेले, धडा चेनच्या स्वतःच्या परंपरेनुसार) अंमलात आणली — जेव्हाही मान्यतेच्या धोरण आवृत्ती, की आणि कालबाह्यता अद्याप वर्तमानात होती, फक्त एकदाच. जर कोणतीही बाजू बदलली, तर चेन बंद पडते आणि नकाराचा कारण तुम्हाला सांगतो **कोणती** मालमत्ता तुटली: जुनी अधिकारता विरुद्ध बदललेली क्रिया.

**सिध्द करत नाही:** की मान्यता UI मनुष्याला जे ते स्वाक्षरी करत असताना ते त्यांना काय समजले होते ते दाखवले (WYSIWYS हा स्वतःचा प्रश्न आहे), की की उलटलेली किंवा चोरलेली नव्हती आधी फिरवण्यापूर्वी, किंवा की पुढील परिणाम क्रियेशी जुळत होते. सही केलेली ≠ अधिकृत: जुनी धोरणावर वैध सहि, फिरवलेली की, कालबाह्य विंडो, किंवा वेगळा डाइजेस्ट येथे काहीही अधिकार देत नाही.

दोन्ही पावत्या प्रकार धड्याची लिफाफा आणि एक `verify_chain` कोड मार्ग उद्देशाने सामायिक करतात: मुख्य नोटबुकमधील क्रिया पावत्या साठी तुम्ही बनवलेली बांधणीच तीच कोड आहे जी मानवी मान्यतेची तपासणी करते. एक पडताळणी करार, वेगळ्या पिन केलेल्या अधिकारांसह, पंचायती क्रिया डाइजेस्टने जोडलेले आणि काहीही नाही.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
